In [7]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import os
import glob

In [2]:
def get_fluences(
    traces,
    delta_t: float,
    pol_axis: int = 0,
    sample_axis: int = 2,
):
    """
    Return the fluences from electric field traces in eV/m^2.

    Parameters
    ----------
    efield_traces : ArrayLike
        the electric field traces
    delta_t : float
        the timing resolution in ns
    pol_axis : int, default=2
        the axis in which the polarisations are located.
    """
    conversion_factor_integrated_signal = 2.65441729e-3 * 6.24150934e18  # to convert V**2/m**2 * s -> J/m**2 -> eV/m**2
    delta_t_s = delta_t * 1e-9
    return (
        conversion_factor_integrated_signal
        * np.sum(
            delta_t_s * np.linalg.norm(traces, axis=pol_axis)**2,
            axis=sample_axis - 1,
        )
    )

In [8]:
files = glob.glob("traces_127417834/*pol0*")

In [10]:
def modify_data(data):
    modified = pd.concat([data.columns.to_frame().T, data]).reset_index(drop=True).iloc[:,0].str.split(expand=True).astype(float)
    return modified.iloc[:,1].values

In [11]:
for i in range(len(files)):
    data = pd.read_csv(files[i], delimiter="\t")
    modified = modify_data(data)

    if i == 0:
        traces = np.empty((len(files), *modified.shape))
    
    traces[i] = modified

In [18]:
fluences

np.float64(29755803.09537693)

In [19]:
fluences = get_fluences(traces, 5, 1)
antenna_positions = np.load("chan_dist.npy") # get this from antenna_positions.hdf5

fluence_cmap = plt.cm.plasma
fluence_norm = mcolors.Normalize(vmin=0, vmax=fluences.max(), clip=True) 

fig, ax = plt.subplots(figsize=(8,6))

sc = ax.scatter(antenna_positions[:,0], antenna_positions[:,1],c=fluences[i], cmap=fluence_cmap, norm=fluence_norm, s=120.0, edgecolor='black', linewidth=0.5, zorder=10)
ax.set_xlabel(r"$x$ / m", fontsize=18)
ax.set_ylabel(r"$y$ / m", fontsize=18)

cbar = fig.colorbar(sc, ax=ax)
cbar.ax.set_ylabel("Fluence (Voltage) / eV m$^{-2}$", fontsize=18)
cbar.ax.tick_params(axis='both', which='major', labelsize=14, size=6)

ax.set_xlim(-400, 400)
ax.set_ylim(-400, 400)

ax.tick_params(axis='both', which='major', labelsize=14, size=6)
ax.tick_params(axis='both', which='minor', labelsize=14, size=4)

# comment out if you dont want to save it
fig.savefig(os.path.join(fig_dir, f"fluence_map.png"), dpi=300, bbox_inches="tight")

AxisError: axis 1 is out of bounds for array of dimension 1